In [1]:
import osmnx as ox
import networkx as nx
import pandas as pd
import re
import os
import matplotlib.pyplot as plt
import callejero as ca
import numpy as np

In [2]:
df = ca.carga_callejero()
df

,VIA_CLASE,VIA_PAR,VIA_NOMBRE,NUMERO,LATITUD,LONGITUD
0,AUTOVIA,,A-1,10000,40.489400,-3.673211
1,AUTOVIA,,A-1,10000,40.489292,-3.672431
2,AUTOVIA,,A-1,11000,40.496703,-3.665981
3,AUTOVIA,,A-1,11000,40.496186,-3.665242
4,AUTOVIA,,A-1,12000,40.503703,-3.659242
...,...,...,...,...,...,...
213806,PASEO,DEL,ZURRON,49,40.469658,-3.584978
213807,PASEO,DEL,ZURRON,51,40.469669,-3.584256
213808,PASEO,DEL,ZURRON,53,40.469817,-3.583908
213809,PASEO,DEL,ZURRON,55,40.469775,-3.583617


In [3]:
ruta_grafo = "madrid.graphml"
existencia = os.path.exists(ruta_grafo)

if not existencia:  #Cargamos el grafo de open street map si no se ha descargado previamente
    G = ox.graph_from_place("Madrid, Spain", network_type="drive")
    ox.save_graphml(G, filepath="madrid.graphml")

G = ox.load_graphml(filepath="madrid.graphml")

In [4]:
G = ox.convert.to_digraph(G, weight="length") #quita aristas repetidas pero no bucles 

loops = list(nx.selfloop_edges(G)) #bucles
G.remove_edges_from(loops)

In [5]:
df_grafo = nx.to_pandas_edgelist(G)

In [6]:
df_grafo

,source,target,name,length,geometry,bridge,oneway,highway,ref,tunnel,est_width,lanes,maxspeed,access,reversed,osmid,junction,width
0,171946,26513145,Calle de Velázquez,43.952,"LINESTRING (-3.6844432 40.4212466, -3.6844412 ...",NaN,True,secondary,NaN,NaN,NaN,4,50,NaN,False,807334397,NaN,NaN
1,171951,1209331009,Calle Juan de Mena,18.733,NaN,NaN,True,residential,NaN,NaN,NaN,NaN,NaN,NaN,False,104864843,NaN,NaN
2,171951,26486636,Calle de Alfonso XII,80.652,"LINESTRING (-3.6889892 40.4173597, -3.6890327 ...",NaN,True,secondary,NaN,NaN,NaN,3,50,NaN,False,553113575,NaN,NaN
3,171952,26486617,Calle de Alfonso XII,26.251,NaN,NaN,True,secondary,NaN,NaN,NaN,3,NaN,NaN,False,317302445,NaN,NaN
4,171953,2681222064,Calle Espalter,18.835,NaN,NaN,True,residential,NaN,NaN,NaN,NaN,NaN,NaN,False,28401583,NaN,NaN
...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...
61308,13287493474,317738337,Paseo de Extremadura,117.849,NaN,NaN,True,unclassified,NaN,NaN,NaN,2,NaN,NaN,False,1239185065,NaN,NaN
61309,13287493474,10199731680,NaN,108.582,"LINESTRING (-3.7925645 40.3777306, -3.7925995 ...",NaN,False,busway,NaN,NaN,NaN,NaN,NaN,NaN,True,1448229224,NaN,NaN
61310,13288899144,21364521,NaN,173.826,NaN,NaN,True,motorway_link,NaN,NaN,NaN,NaN,NaN,NaN,False,1294960436,NaN,NaN
61311,13288899144,13288899145,NaN,211.240,NaN,NaN,True,motorway_link,NaN,NaN,NaN,NaN,NaN,NaN,False,1448391667,NaN,NaN


In [16]:
serie = df_grafo[df_grafo["name"] == "Calle de Alfonso XII"]["geometry"]
list(serie.iloc[0].coords)

[(-3.6889892, 40.4173597),
 (-3.6890327, 40.4168733),
 (-3.6890333, 40.4168667),
 (-3.6890546, 40.4166361)]

In [77]:
MAX_SPEEDS={'living_street': 20,
 'residential': 30,
 'primary_link': 40,
 'unclassified': 40,
 'secondary_link': 40,
 'trunk_link': 40,
 'secondary': 50,
 'tertiary': 50,
 'primary': 50,
 'trunk': 50,
 'tertiary_link':50,
 'busway': 50,
 'motorway_link': 70,
 'motorway': 100}

In [79]:
def funcion(serie):
    if isinstance(serie["highway"], list):
        a = -1
        b = "tipo"
        for i in serie["highway"]:
            val = MAX_SPEEDS[i]
            if val > a:
                a = val
                b = i
        serie["highway"] = b
        serie["maxspeed"] = a
    return serie



df_grafo = df_grafo.apply(func=funcion, axis=1)

In [80]:
#Highway ya lo tienes ready no tiene ni listas ni NaN ahora te toca cambiar los maxspeed para los que o no hay o son una lista, pones directamente el del highway.
df_grafo["maxspeed"] = df_grafo["highway"].map(MAX_SPEEDS)


In [81]:
df_grafo["time"] = df_grafo["length"] / (df_grafo["maxspeed"] * 1_000)

In [82]:
df_grafo.head(3)

,source,target,maxspeed,highway,tunnel,geometry,oneway,bridge,access,est_width,ref,width,osmid,name,junction,lanes,reversed,length,time
0,171946,26513145,50,secondary,NaN,"LINESTRING (-3.6844432 40.4212466, -3.6844412 ...",True,NaN,NaN,NaN,NaN,NaN,807334397,Calle de Velázquez,NaN,4,False,43.952,0.000879
1,171951,1209331009,30,residential,NaN,NaN,True,NaN,NaN,NaN,NaN,NaN,104864843,Calle Juan de Mena,NaN,NaN,False,18.733,0.000624
2,171951,26486636,50,secondary,NaN,"LINESTRING (-3.6889892 40.4173597, -3.6890327 ...",True,NaN,NaN,NaN,NaN,NaN,553113575,Calle de Alfonso XII,NaN,3,False,80.652,0.001613


In [83]:
df_grafo.columns

Index(['source', 'target', 'maxspeed', 'highway', 'tunnel', 'geometry',
       'oneway', 'bridge', 'access', 'est_width', 'ref', 'width', 'osmid',
       'name', 'junction', 'lanes', 'reversed', 'length', 'time'],
      dtype='object')

In [ ]:
for u, v, data in G.edges(data=True):
    if isinstance(data["highway"], list):
        a = -1
        b = "tipo"
        for i in data["highway"]:
            val = MAX_SPEEDS[i]
            if val > a:
                a = val
                b = i
            data["highway"] = b
            data["maxspeed"] = a
    else:
        data["maxspeed"] = MAX_SPEEDS[data["highway"]]
    data["time"] = data["length"] / (data["maxspeed"] * 1_000)